# Coleta de PVs

## Introdução

Esse notebook tem como objetivo realizar a coleta de dados de diferentes PVs (Variáveis de Processos) no tempo determinado pelos logs registrados.

Para isso, iremos utilizar a API do EPICS Archiver [[1](#referencias)], o sistema que armazena e organiza as informações de PVs do Sirius.

As principais PVs que iremos utilizar no nosso projeto são as do `sismógrafo` e a dos `BPMs` (Beam Position Monitors).

## Desenvolvimento

### Função para recuperar os dados da API

Vamos começar escrevendo uma função que busca os dados da API. O link base é `https://ais-eng-srv-ca.cnpem.br/retrieval/data/getData.json`, e os parâmetros que conseguimos passar pela url são: *pv* (o identificador da pv que queremos), *from* (o tempo de início) e *to* (o tempo final).

Ao passar o identificador da pv, também podemos utilizar a função optmize junto de algum valor, que irá diminuir o número de amostras capturadas para o valor determinado, agrupando amostras parecidas.

In [ ]:
import matplotlib.pyplot as plt
import requests
import pandas as pd
import datetime
import os

def get_archiver_data(
    pv,
    from_time,
    to_time,
    optimize=-1,
    diff=True,
    base_url="https://ais-eng-srv-ca.cnpem.br/retrieval/data/getData.json"
):
    """
    Retrieve time-series data from the EPICS Archiver Appliance and return it
    as a pandas DataFrame.

    Parameters
    ----------
    pv : str
        Process Variable (PV) name to query.
    from_time : str
        Start timestamp of the query interval. Must be in a format accepted
        by the Archiver Appliance API.
    to_time : str
        End timestamp of the query interval. Must be in a format accepted
        by the Archiver Appliance API.
    optimize : int, optional
        Optimization level for server-side aggregation. If set to -1, raw
        data are retrieved. Otherwise, the PV is requested using the
        ``optimized_<optimize>(PV)`` syntax provided by the Archiver API.
        Default is -1.
    diff : bool, optional
        If ``True`` and an optimized query is used, the returned ``val``
        column is shifted so that its first value becomes zero. This is
        useful for analyzing cumulative quantities as relative variations
        over the selected time interval. Ignored when ``optimize = -1``.
        Default is ``True``.
    base_url : str, optional
        URL of the Archiver Appliance data retrieval endpoint.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing the retrieved data.

        For non-optimized queries, the DataFrame contains the fields
        returned directly by the API.

        For optimized queries, the aggregated statistics are expanded into
        separate columns:

        - ``val``: aggregated value (or relative value if ``diff=True``)
        - ``std``: standard deviation
        - ``min``: minimum value
        - ``max``: maximum value
        - ``count``: number of samples used in the aggregation

    Raises
    ------
    requests.HTTPError
        If the HTTP request to the Archiver Appliance fails.

    Notes
    -----
    The function expects the response format of the Archiver Appliance
    ``getData.json`` endpoint, where the payload is available at
    ``response.json()[0]["data"]``.

    When optimization is enabled, the API returns each value as a list
    containing the aggregated value and additional statistics. These are
    automatically unpacked into separate DataFrame columns.
    """
    
    parameters = {
        "pv": f"optimized_{optimize}({pv})" if optimize != -1 else pv,
        "from": from_time,
        "to": to_time
    }
    
    response = requests.get(base_url, params=parameters)
    
    if not response.ok:
        response.raise_for_status()
        
    json_data = response.json()[0]['data']
    
    df = pd.json_normalize(json_data)
    
    if optimize != -1:
        df["std"]   = df["val"].apply(lambda x: x[1])
        df["min"]   = df["val"].apply(lambda x: x[2])
        df["max"]   = df["val"].apply(lambda x: x[3])
        df["count"] = df["val"].apply(lambda x: int(x[4]))
        
        df["val"]   = df["val"].apply(lambda x: x[0])
        if diff: df["val"] -= df["val"][0]
    
    return df

Podemos testar a função recebendo os dados de uma das PVs de sismógrafo do eixo Z entre as 15h00 e as 17h00 do dia 09/10/2026 (período da interferência causada pelo terromoto do Panamá).

Nesse caso, como vamos plotar o sinal obtido, podemos utilizar o optimize como 800 pois não precisamos de uma definição tão alta.

In [ ]:
df = get_archiver_data(
    pv="SI-01C1:DI-BPM-1:PosX-Mon",
    from_time="2026-10-09T18:00:00.000000Z",
    to_time="2026-10-09T20:00:00.000000Z",
    optimize=800
)

df

Uma vez com os dados em mãos, conseguimos plotá-los com o matplotlib.

In [ ]:
plt.figure(figsize=(12,10))
plt.plot(df["secs"], df["val"])
plt.ylim(-20000, 20000)

plt.xlabel("Segundos")
plt.ylabel("Sinal")
plt.title("Sismógrafo Eixo Z (RR-09S:SS-Seism-Ax13:SZ-Mon)")

plt.show()

## Capturando os dados dos logs

Agora que temos a função para consultar a API do archiver, podemos capturar os dados obtidos anteriormente pelos logs para conseguir processar toda essa informação de uma forma melhor posteriormente.

Primeiro, vamos ler o arquivo gerado:

In [ ]:
olog_data = pd.read_csv("olog_data.csv")
olog_data

Vamos precisar de algumas funções de suporte para lidar com o padrão de tempo utilizado no arquivo

In [ ]:
def add_hours(timestamp, hours):
    return (datetime.datetime.strptime(timestamp, "%Y-%m-%dT%H:%M:%S.%fZ") + datetime.timedelta(hours=hours)).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

def sub_hours(timestamp, hours):
    return (datetime.datetime.strptime(timestamp, "%Y-%m-%dT%H:%M:%S.%fZ") - datetime.timedelta(hours=hours)).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

def seconds_between(from_time, to_time):
    return datetime.datetime.strptime(from_time, "%Y-%m-%dT%H:%M:%S.%fZ") - datetime.datetime.strptime(to_time, "%Y-%m-%dT%H:%M:%S.%fZ").total_seconds()

Agora podemos gerar um novo arquivo para cada um dos logs presentes.

*Nota:* Nem todos os logs encontram corretamente os dados. Provavelmente por certos eventos não terem todas as informações registradas nas PVs consultadas. Esses arquivos devem ser deletados após a coleta inicial.

In [ ]:
error_ids = []

for i, log in olog_data.iterrows():
    filename = f"sismic_data/{log['id']}.xlsx"
    
    print(f"LOG {i:2} - {log["id"]}: Procurando dados...")

    try:
        with pd.ExcelWriter(filename, engine="openpyxl") as writer:
            
            from_time = sub_hours(log["from"], 2)
            to_time = add_hours(log["to"], 2)
        
            for axis in ["Z", "X", "Y"]:                
                
                df = get_archiver_data(
                    pv=f"RR-09S:SS-Seism-Ax13:S{axis}-Mon",
                    from_time=from_time,
                    to_time=to_time,
                    optimize=800
                )
                
                df.to_excel(
                    writer,
                    sheet_name=f"S{axis}",
                    index=False
                )
            
            for axis in ["X", "Y"]:
                        
                        df = get_archiver_data(
                            pv=f"SI-01C1:DI-BPM-1:Pos{axis}-Mon",
                            from_time=from_time,
                            to_time=to_time,
                            optimize=800
                        )
                        
                        df.to_excel(
                            writer,
                            sheet_name=f"BPM{axis}",
                            index=False
                        )
                        
                
        print(f"LOG {i:2} - {log["id"]}: Processamento Concluído!")
    except:
        print(f"LOG {i:2} - {log["id"]}: Erro no processamento")
        error_ids.append(log["id"])

E, para deletar:

In [ ]:
for id in error_ids:
    filename = f"sismic_data/{id}.xlsx"
    
    if os.path.exists(filename):
        os.remove(filename)

## Conclusões

Agora temos diversas planilhas com os registros de sismógrafo e BPM de diferentes terremotos. O próximo passo será calcular as ondas superficiais e coróreas geradas, para então identificar a correlação entre cada onda com a interferência causada nos feixes.

<span id="referencias"></span>
## <a id='toc1_4_'></a>[Referências](#toc0_)

1. EPICS ARCHIVER. *User guide: retrieving data using other tools*. [S. l.]: EPICS Archiver Documentation, [s. d.]. Disponível em: https://epicsarchiver.readthedocs.io/en/stable/user/userguide.html#retrieving-data-using-other-tools. Acesso em: 26 out. 2026.